# 07 — TabDDPM: Mixed-Type Diffusion for Synthetic Clinical Tabular Data

**Capstone:** Comparative Evaluation of Diffusion, GAN, and VAE Models for Synthetic Clinical Tabular Data Generation  
**Input:** `data/processed/train_semantic.csv` (patient-disjoint training partition only)  
**Output:** 79,567 synthetic rows with the same 45 columns

### Method note — read before running

This is a **self-contained, TabDDPM-inspired *joint/unconditional* implementation**, not a byte-for-byte execution of the authors' official repository. It implements Gaussian diffusion for the **8 continuous/count features** and multinomial diffusion for **37 categorical features including the binary target**, using a time-conditioned MLP and a Gaussian MSE + categorical variational KL objective. The original paper typically **conditions on the class label** for classification datasets; here we **model `readmitted_30` jointly as a categorical variable** so the model must learn its prevalence, rather than receiving the real 11.40% rate at generation time. **Document this design choice in the report.**

The model uses cosine noise scheduling, quantile normalization for numerical fields, empirical categories learned only from training data, and exact categorical forward/posterior kernels. This notebook is intended as a reproducible **research implementation** and should be reported as such, not as an exact reproduction of the authors' benchmark configuration.

Paper: Kotelnikov et al. (2023), *TabDDPM: Modelling Tabular Data with Diffusion Models*, ICML — https://proceedings.mlr.press/v202/kotelnikov23a.html  
Official reference code: https://github.com/yandex-research/tab-ddpm

**Important:** Execute cells in order. Do **not** use *Run All* again after training is complete unless you intend to repeat the experiment. Training is resumable from a checkpoint, but long-running Jupyter cells can still stop if the remote session disconnects.

## 1. Setup, configuration, and data checks

Uses the project's existing `.venv` kernel on the DGX A100; no new packages should be necessary beyond PyTorch, pandas, NumPy, and scikit-learn.

In [1]:
from pathlib import Path
import os, math, json, random, time, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import QuantileTransformer
from IPython.display import display, Markdown

SEED = 42
EPOCHS = 100                 # Baseline run; adjust only intentionally after baseline is saved
DIFFUSION_STEPS = 200        # Sampling takes one reverse pass per step per batch
BATCH_SIZE = 1024
LEARNING_RATE = 5e-4
HIDDEN_DIM = 512
DEPTH = 3
DROPOUT = 0.05
SAMPLE_BATCH_SIZE = 1024
ROWS_PER_SAVED_CHUNK = 4096

# Detect root whether VS Code opens the project directory or its notebooks subdirectory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/train_semantic.csv').is_file()), None)
assert ROOT is not None, 'Open VS Code in synthetic-clinical-data-generation or its notebooks directory.'
DATA_PATH = ROOT / 'data/processed/train_semantic.csv'
META_PATH = ROOT / 'data/processed/preprocessing_metadata.json'
NAME = f'tabddpm_joint_{EPOCHS}ep'
CKPT_PATH = ROOT / 'models' / f'{NAME}_latest.pt'
FINAL_PATH = ROOT / 'models' / f'{NAME}.pt'
OUT_PATH = ROOT / 'data/synthetic' / f'{NAME}.csv'
RUN_PATH = ROOT / 'results/training' / f'{NAME}_run.json'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Training this full model requires the DGX CUDA GPU.'
print('Project:', ROOT)
print('Device:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Checkpoint:', CKPT_PATH.relative_to(ROOT))

Project: /home/ayanasanthosh.ds25a/synthetic-clinical-data-generation
Device: NVIDIA A100-SXM4-80GB
PyTorch: 2.3.1+cu121
Checkpoint: models/tabddpm_joint_100ep_latest.pt


## 2. Training-only preprocessing

Adopts the existing patient-disjoint split. The preprocessing metadata specifies which numeric-looking code columns are **categorical**. We fit the numerical transform on the training set only; the held-out test set is **never used for training or transformation fitting**.

In [2]:
real = pd.read_csv(DATA_PATH)
with open(META_PATH, 'r', encoding='utf-8') as f:
    existing_meta = json.load(f)
num_cols = list(existing_meta['numerical_feature_columns'])
cat_cols = list(existing_meta['categorical_feature_columns']) + ['readmitted_30']
assert len(real) == 79567 and len(real.columns) == 45, f'Unexpected input shape: {real.shape}'
assert len(num_cols) == 8 and len(cat_cols) == 37
assert set(num_cols + cat_cols) == set(real.columns)
assert not real[num_cols].isna().any().any(), 'Unexpected missing numeric values'
assert not real[cat_cols].isna().any().any(), 'Unexpected missing categorical values'
assert real['readmitted_30'].isin([0,1]).all()

# This fingerprint guards against accidentally resuming an incompatible model.
def sha256_file(path, chunk_bytes=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_bytes), b''): h.update(chunk)
    return h.hexdigest()

data_sha = sha256_file(DATA_PATH)
transformer = QuantileTransformer(n_quantiles=min(1000, len(real)), output_distribution='normal', random_state=SEED)
num_np = transformer.fit_transform(real[num_cols].to_numpy(dtype='float64')).astype('float32')
cat_levels = [list(pd.unique(real[c].astype(str))) for c in cat_cols]
cat_np = np.stack([
    pd.Categorical(real[c].astype(str), categories=levels).codes.astype('int64')
    for c, levels in zip(cat_cols, cat_levels)
], axis=1)
assert cat_np.min() >= 0
K = [len(levels) for levels in cat_levels]
D_NUM = len(num_cols)
D_CAT = len(cat_cols)
D_ONEHOT = sum(K)
num_data = torch.from_numpy(num_np)
cat_data = torch.from_numpy(cat_np)
print('Real training rows:', len(real))
print('Numerical columns:', D_NUM, '| categorical incl target:', D_CAT)
print('One-hot categorical width:', D_ONEHOT, '| model output width:', D_NUM + D_ONEHOT)
print('Original readmission counts:', real['readmitted_30'].value_counts().to_dict())
print('Dataset SHA256:', data_sha)

Real training rows: 79567
Numerical columns: 8 | categorical incl target: 37
One-hot categorical width: 249 | model output width: 257
Original readmission counts: {0: 70497, 1: 9070}
Dataset SHA256: fcd79b513af77bed22967520e161e56d3cc9f38f7bc445fe45346845f1aba382


## 3. Model: time-conditioned MLP + mixed Gaussian/multinomial diffusion

One denoiser predicts Gaussian noise for numerical columns and logits for the original category of each discrete column. For categorical variables, the diffusion posterior is computed from the paper's uniform-transition multinomial process; the loss uses posterior KL (plus the reconstruction likelihood at the final reverse step).

In [3]:
class TabDenoiser(nn.Module):
    def __init__(self, n_num, cardinals, hidden=512, depth=3, dropout=0.05):
        super().__init__()
        self.n_num = int(n_num)
        self.cardinals = [int(k) for k in cardinals]
        d = self.n_num + sum(self.cardinals)
        self.input_proj = nn.Linear(d, hidden)
        self.time_mlp = nn.Sequential(nn.Linear(128, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.blocks = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Dropout(dropout))
            for _ in range(depth)
        ])
        self.output_proj = nn.Linear(hidden, d)

    def forward(self, xnum_t, xcat_t, step):
        enc = [F.one_hot(xcat_t[:, j], num_classes=k).to(xnum_t.dtype)
               for j, k in enumerate(self.cardinals)]
        x = torch.cat([xnum_t, *enc], dim=-1)
        half = 64
        scale = torch.exp(-math.log(10000.0) * torch.arange(half, device=x.device) / (half-1))
        angles = step[:, None].float() * scale[None, :]
        t_embed = torch.cat([angles.sin(), angles.cos()], dim=1)
        h = self.input_proj(x) + self.time_mlp(t_embed)
        for block in self.blocks:
            h = h + 0.5 * block(h)
        out = self.output_proj(h)
        return out[:, :self.n_num], out[:, self.n_num:].split(self.cardinals, dim=1)


class MixedTabDDPM:
    """Mixed Gaussian + multinomial diffusion with independent categorical kernels."""
    def __init__(self, n_steps, device):
        self.T = int(n_steps)
        self.device = device
        # Cosine schedule: alpha_bar near zero at the final step (good generation prior).
        s = 0.008
        steps = torch.arange(self.T + 1, dtype=torch.float64)
        cum = torch.cos(((steps / self.T + s)/(1+s))*math.pi/2)**2
        cum = cum / cum[0]
        beta = (1 - cum[1:]/cum[:-1]).clamp(min=1e-5, max=0.999).float().to(device)
        self.beta = beta
        self.alpha = 1 - beta
        self.ab = torch.cumprod(self.alpha, dim=0)
        self.ab_prev = torch.cat([torch.ones(1,device=device), self.ab[:-1]])
        self.posterior_variance = beta * (1-self.ab_prev) / (1-self.ab).clamp(min=1e-12)

    def training_loss(self, net, xnum, xcat):
        b = xnum.shape[0]
        t = torch.randint(0, self.T, (b,), device=xnum.device)
        a_bar = self.ab[t]
        eps = torch.randn_like(xnum)
        noisy_num = a_bar.sqrt()[:,None]*xnum + (1-a_bar).sqrt()[:,None]*eps
        noisy_categories = []
        for j, k in enumerate(net.cardinals):
            keep = torch.rand(b, device=xnum.device) < a_bar
            random_cat = torch.randint(k, (b,), device=xnum.device)
            noisy_categories.append(torch.where(keep, xcat[:,j], random_cat))
        noisy_cat = torch.stack(noisy_categories,dim=1)
        pred_eps, logits = net(noisy_num, noisy_cat, t)
        gaussian_mse = F.mse_loss(pred_eps, eps)

        losses = []
        for j, (k, logit) in enumerate(zip(net.cardinals, logits)):
            true_x0 = F.one_hot(xcat[:,j], num_classes=k).float()
            noisy_onehot = F.one_hot(noisy_cat[:,j], num_classes=k).float()
            # q(x_previous|x_t, x0) proportional to q(x_t|x_previous)q(x_previous|x0).
            transition = self.alpha[t,None] * noisy_onehot + (1-self.alpha[t,None]) / k
            prior_true = self.ab_prev[t,None] * true_x0 + (1-self.ab_prev[t,None]) / k
            unnorm_true = transition * prior_true
            qpost = unnorm_true / unnorm_true.sum(dim=-1, keepdim=True).clamp(min=1e-20)
            predicted_x0 = F.softmax(logit, dim=-1)
            prior_pred = self.ab_prev[t,None] * predicted_x0 + (1-self.ab_prev[t,None]) / k
            unnorm_pred = transition * prior_pred
            ppost = unnorm_pred / unnorm_pred.sum(dim=-1, keepdim=True).clamp(min=1e-20)
            kl = (qpost * (qpost.clamp(min=1e-20).log() - ppost.clamp(min=1e-20).log())).sum(-1)
            nll = F.cross_entropy(logit, xcat[:,j], reduction='none')
            losses.append(torch.where(t==0, nll, kl))
        # Monte-Carlo estimate of the *sum over diffusion steps* / categorical features.
        categorical_vlb = torch.stack(losses,dim=1).mean() * self.T
        return gaussian_mse + categorical_vlb, gaussian_mse.detach(), categorical_vlb.detach()

    @torch.no_grad()
    def sample(self, net, size, device):
        net.eval()
        xnum = torch.randn((size,net.n_num),device=device)
        xcat = torch.stack([torch.randint(k,(size,),device=device)
                            for k in net.cardinals],dim=1)
        for t in range(self.T-1,-1,-1):
            steps = torch.full((size,),t,device=device,dtype=torch.long)
            eps, logits = net(xnum,xcat,steps)
            # Standard DDPM Gaussian posterior using predicted x0.
            predicted_x0 = (xnum - (1-self.ab[t]).sqrt()*eps) / self.ab[t].sqrt().clamp(min=1e-8)
            predicted_x0 = predicted_x0.clamp(-5.5,5.5)
            coef1 = self.beta[t] * self.ab_prev[t].sqrt() / (1-self.ab[t]).clamp(min=1e-12)
            coef2 = self.alpha[t].sqrt() * (1-self.ab_prev[t]) / (1-self.ab[t]).clamp(min=1e-12)
            mean = coef1*predicted_x0 + coef2*xnum
            xnum = mean if t == 0 else mean + self.posterior_variance[t].sqrt()*torch.randn_like(xnum)
            next_cats=[]
            for j,(k,logit) in enumerate(zip(net.cardinals, logits)):
                predicted = F.softmax(logit,dim=-1)
                if t == 0:
                    probs = predicted
                else:
                    onehot_xt = F.one_hot(xcat[:,j],num_classes=k).float()
                    transition = self.alpha[t]*onehot_xt + (1-self.alpha[t])/k
                    prior = self.ab_prev[t]*predicted + (1-self.ab_prev[t])/k
                    prob = transition*prior
                    probs = prob / prob.sum(dim=1,keepdim=True).clamp(min=1e-20)
                # Sample the posterior categorical distribution; no forced class proportions.
                next_cats.append(torch.multinomial(probs,1).squeeze(1))
            xcat = torch.stack(next_cats,dim=1)
        return xnum.cpu().numpy(), xcat.cpu().numpy()


def make_model():
    return TabDenoiser(D_NUM, K, HIDDEN_DIM, DEPTH, DROPOUT).to(DEVICE)

net = make_model()
diffusion = MixedTabDDPM(DIFFUSION_STEPS, DEVICE)
print('Model parameters:', sum(p.numel() for p in net.parameters()))
print('Final numerical alpha_bar:', diffusion.ab[-1].item())

Model parameters: 1380609
Final numerical alpha_bar: 6.071716995847964e-08


## 4. Small smoke test (no training)

Tests one forward/backward pass before investing GPU time. It does **not** update weights, so rerunning this cell cannot corrupt a saved model.

In [4]:
net.train()
net.zero_grad(set_to_none=True)
loss_test, num_test, cat_test = diffusion.training_loss(net, num_data[:64].to(DEVICE), cat_data[:64].to(DEVICE))
assert torch.isfinite(loss_test), f'Non-finite loss: {loss_test.item()}'
loss_test.backward()
assert all(torch.isfinite(p.grad).all() for p in net.parameters() if p.grad is not None)
net.zero_grad(set_to_none=True)
print(f'Smoke test PASSED | total={loss_test.item():.4f} | numeric={num_test.item():.4f} | categorical={cat_test.item():.4f}')

Smoke test PASSED | total=6.9594 | numeric=0.9413 | categorical=6.0181


## 5. Train and checkpoint every epoch

**Run this once.** It prints an epoch-by-epoch record and saves a resume checkpoint after *every completed epoch*. If VS Code disconnects, reopen this notebook, rerun sections 1–3 and then this cell to continue. The final saved model is separate from the rolling checkpoint. Never delete an existing final model.

In [5]:
for folder in [CKPT_PATH.parent, OUT_PATH.parent, RUN_PATH.parent]: folder.mkdir(parents=True,exist_ok=True)
config = dict(seed=SEED, total_epochs=EPOCHS, diffusion_steps=DIFFUSION_STEPS,
              batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
              hidden_dim=HIDDEN_DIM, depth=DEPTH, dropout=DROPOUT,
              data_sha256=data_sha, num_cols=num_cols, cat_cols=cat_cols,
              cat_levels=cat_levels, target_model='joint_unconditional')
optimizer = torch.optim.AdamW(net.parameters(),lr=LEARNING_RATE,weight_decay=0.0)
loader_rng = torch.Generator().manual_seed(SEED)
loader = DataLoader(TensorDataset(num_data, cat_data),batch_size=BATCH_SIZE,
                    shuffle=True,num_workers=0,generator=loader_rng)
loss_history = []
start_epoch = 0
if CKPT_PATH.exists():
    resume = torch.load(CKPT_PATH,map_location=DEVICE,weights_only=False)
    assert resume['config'] == config, 'Checkpoint configuration/data mismatch; do not overwrite it.'
    net.load_state_dict(resume['model'])
    optimizer.load_state_dict(resume['optimizer'])
    start_epoch = int(resume['epoch'])
    loss_history = resume['history']
    torch.set_rng_state(resume['torch_rng'])
    torch.cuda.set_rng_state(resume['cuda_rng'])
    loader_rng.set_state(resume['loader_rng'])
    print('Resuming after completed epoch:',start_epoch)
elif FINAL_PATH.exists():
    saved = torch.load(FINAL_PATH,map_location=DEVICE,weights_only=False)
    assert saved['config'] == config, 'Saved final model belongs to another configuration.'
    net.load_state_dict(saved['model'])
    start_epoch = EPOCHS
    loss_history = saved['history']
    print('Full model already saved; training will be skipped.')

begin = time.time()
for epoch in range(start_epoch,EPOCHS):
    net.train()
    total,gaussian,categorical,count = 0.0,0.0,0.0,0
    for nb, cb in loader:
        nb,cb = nb.to(DEVICE,non_blocking=True), cb.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss,gn,cat = diffusion.training_loss(net,nb,cb)
        if not torch.isfinite(loss): raise RuntimeError(f'Nonfinite loss at epoch {epoch+1}')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),max_norm=1.0)
        optimizer.step()
        b = len(nb);total+=loss.item()*b; gaussian+=gn.item()*b; categorical+=cat.item()*b;count+=b
    report = dict(epoch=epoch+1,loss=total/count,gaussian=gaussian/count,categorical=categorical/count)
    loss_history.append(report)
    snapshot = dict(model=net.state_dict(), optimizer=optimizer.state_dict(),
                    epoch=epoch+1,history=loss_history,config=config,
                    torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state(),
                    loader_rng=loader_rng.get_state())
    temp_path = CKPT_PATH.with_suffix('.tmp')
    torch.save(snapshot,temp_path)
    os.replace(temp_path,CKPT_PATH)    # Atomic rename; prior checkpoint remains if save fails.
    print(f"Epoch {epoch+1:03d}/{EPOCHS} | total {report['loss']:.4f} | "
          f"Gaussian {report['gaussian']:.4f} | categorical {report['categorical']:.4f}",flush=True)

if not FINAL_PATH.exists():
    final = dict(model=net.state_dict(), config=config,history=loss_history,
                 epoch=EPOCHS,arch='joint_mixed_gaussian_multinomial_ddpm')
    tmp_final = FINAL_PATH.with_suffix('.tmp')
    torch.save(final,tmp_final)
    os.replace(tmp_final,FINAL_PATH)
print('TRAINING COMPLETED / SAVED:', FINAL_PATH.relative_to(ROOT))

Epoch 001/100 | total 2.1628 | Gaussian 0.7961 | categorical 1.3667
Epoch 002/100 | total 1.8328 | Gaussian 0.6533 | categorical 1.1795
Epoch 003/100 | total 1.6239 | Gaussian 0.5971 | categorical 1.0268
Epoch 004/100 | total 1.5127 | Gaussian 0.5392 | categorical 0.9734
Epoch 005/100 | total 1.4587 | Gaussian 0.5125 | categorical 0.9462
Epoch 006/100 | total 1.3717 | Gaussian 0.4918 | categorical 0.8799
Epoch 007/100 | total 1.3386 | Gaussian 0.4719 | categorical 0.8667
Epoch 008/100 | total 1.2980 | Gaussian 0.4563 | categorical 0.8417
Epoch 009/100 | total 1.2783 | Gaussian 0.4467 | categorical 0.8316
Epoch 010/100 | total 1.2304 | Gaussian 0.4363 | categorical 0.7941
Epoch 011/100 | total 1.2200 | Gaussian 0.4304 | categorical 0.7897
Epoch 012/100 | total 1.1788 | Gaussian 0.4240 | categorical 0.7548
Epoch 013/100 | total 1.1595 | Gaussian 0.4201 | categorical 0.7394
Epoch 014/100 | total 1.1418 | Gaussian 0.4160 | categorical 0.7258
Epoch 015/100 | total 1.1144 | Gaussian 0.4108 |

## 6. Generate synthetic data (interrupt-safe chunks)

Draws all numerical and categorical fields **jointly**; `readmitted_30` is **not manually balanced**. Each completed chunk is saved to disk, so a notebook disconnection need not waste all sampling. Re-running this cell skips previously completed chunks generated with the **same model checksum**. Final output matches the original column order and types.

In [6]:
assert FINAL_PATH.exists(), 'Complete training first; final model is not yet saved.'
final_saved = torch.load(FINAL_PATH,map_location=DEVICE,weights_only=False)
assert final_saved['config'] == config, 'Final model does not match this dataset/configuration.'
net.load_state_dict(final_saved['model']); net.eval()
model_sha = sha256_file(FINAL_PATH)
parts_dir = ROOT/'results/training'/f'{NAME}_chunks_{model_sha[:12]}'
parts_dir.mkdir(parents=True,exist_ok=True)

num_bounds = [(float(real[c].min()), float(real[c].max())) for c in num_cols]
def decode_batch(xnum_norm,xcat_ids):
    original_nums = transformer.inverse_transform(xnum_norm.astype('float64'))
    d={}
    for j,c in enumerate(num_cols):
        lo,hi = num_bounds[j]
        vals=np.rint(np.clip(original_nums[:,j],lo,hi)).astype(real[c].dtype)
        d[c]=vals
    for j,c in enumerate(cat_cols):
        values=np.asarray(cat_levels[j],dtype=object)[xcat_ids[:,j]]
        if pd.api.types.is_integer_dtype(real[c].dtype):
            values=values.astype(real[c].dtype)
        d[c]=values
    return pd.DataFrame(d)[list(real.columns)]

parts=[]
for start in range(0,len(real),ROWS_PER_SAVED_CHUNK):
    part_path=parts_dir/f'chunk_{start:06d}.csv'
    n_chunk=min(ROWS_PER_SAVED_CHUNK,len(real)-start)
    if part_path.exists():
        old_rows=sum(1 for _ in open(part_path,encoding='utf-8'))-1
        if old_rows == n_chunk:
            print(f'Reusing existing chunk {start:,}–{start+n_chunk-1:,}')
            parts.append(part_path)
            continue
        raise RuntimeError(f'Incomplete chunk {part_path}; inspect before overwriting')
    batches=[]
    for pos in range(0,n_chunk,SAMPLE_BATCH_SIZE):
        nnow=min(SAMPLE_BATCH_SIZE,n_chunk-pos)
        xn,xc=diffusion.sample(net,nnow,DEVICE)
        batches.append(decode_batch(xn,xc))
    one_chunk=pd.concat(batches,ignore_index=True)
    tmp=part_path.with_suffix('.tmp')
    one_chunk.to_csv(tmp,index=False)
    os.replace(tmp,part_path)
    parts.append(part_path)
    print(f'Saved chunk {start:,}–{start+n_chunk-1:,}',flush=True)

# Combine only after every chunk exists; no partial synthetic output is mistaken for final.
assert sum(len(pd.read_csv(p)) for p in parts)==len(real)
synthetic=pd.concat([pd.read_csv(p) for p in parts],ignore_index=True)
synthetic=synthetic[list(real.columns)]
assert synthetic.shape==real.shape
assert synthetic['readmitted_30'].isin([0,1]).all()
tmp_output=OUT_PATH.with_suffix('.tmp')
synthetic.to_csv(tmp_output,index=False)
os.replace(tmp_output,OUT_PATH)
print('SYNTHETIC DATA SAVED:',OUT_PATH.relative_to(ROOT),'rows=',len(synthetic))

Saved chunk 0–4,095
Saved chunk 4,096–8,191
Saved chunk 8,192–12,287
Saved chunk 12,288–16,383
Saved chunk 16,384–20,479
Saved chunk 20,480–24,575
Saved chunk 24,576–28,671
Saved chunk 28,672–32,767
Saved chunk 32,768–36,863
Saved chunk 36,864–40,959
Saved chunk 40,960–45,055
Saved chunk 45,056–49,151
Saved chunk 49,152–53,247
Saved chunk 53,248–57,343
Saved chunk 57,344–61,439
Saved chunk 61,440–65,535
Saved chunk 65,536–69,631
Saved chunk 69,632–73,727
Saved chunk 73,728–77,823
Saved chunk 77,824–79,566
SYNTHETIC DATA SAVED: data/synthetic/tabddpm_joint_100ep.csv rows= 79567


## 7. Initial quality checks and interpretation

These are **descriptive checks**, not yet a full fidelity, TSTR utility, or privacy evaluation. Compare against all models later using identical downstream protocols. Pay particular attention to the 30-day readmission minority class and count-variable distributions.

In [7]:
assert OUT_PATH.exists(), 'Run sampling (Section 6) before diagnostics.'
synth=pd.read_csv(OUT_PATH)
assert synth.shape==real.shape
pos_real=int(real['readmitted_30'].eq(1).sum())
pos_synth=int(synth['readmitted_30'].eq(1).sum())
print('READMISSION 30-DAY LABEL DISTRIBUTION')
summary=pd.DataFrame({'Count':[len(real),len(synth)],'Positive':[pos_real,pos_synth],
                      'Positive rate %':[100*pos_real/len(real),100*pos_synth/len(synth)]},
                     index=['Real train','TabDDPM synthetic'])
display(summary.round(3))
print('\nMEAN PRIOR VISITS')
visit_cols=['number_outpatient','number_emergency','number_inpatient']
display(pd.DataFrame({'Real':real[visit_cols].mean(),
                      'TabDDPM':synth[visit_cols].mean()}).round(3))
print('\nCATEGORY COVERAGE')
for col in ['race','diag_3','readmitted_30']:
    print(f'{col}: real={real[col].nunique()}, TabDDPM={synth[col].nunique()}')

results=dict(model=NAME,model_type='TabDDPM-inspired joint/unconditional',
             data_sha256=data_sha,model_sha256=model_sha,epochs=EPOCHS,
             steps=DIFFUSION_STEPS,n_real=len(real),n_synthetic=len(synth),
             real_positive=pos_real,synthetic_positive=pos_synth,
             real_positive_pct=100*pos_real/len(real),
             synthetic_positive_pct=100*pos_synth/len(synth),
             model_path=str(FINAL_PATH.relative_to(ROOT)),
             synthetic_path=str(OUT_PATH.relative_to(ROOT)),
             training_loss_history=loss_history)
with open(RUN_PATH,'w',encoding='utf-8') as f: json.dump(results,f,indent=2)
print('\nRun metadata saved:',RUN_PATH.relative_to(ROOT))

positive_gap=100*(pos_synth/len(synth)-pos_real/len(real))
display(Markdown(
    f'**Initial interpretation:** TabDDPM generated **{pos_synth:,} positive readmission cases** '
    f'({100*pos_synth/len(synth):.3f}%), against **{pos_real:,}** '
    f'({100*pos_real/len(real):.3f}%) in real training data; the gap is '
    f'**{positive_gap:+.3f} percentage points**. The target was generated jointly, '
    'not sampled from a fixed class prior. This alone does **not** establish fidelity, '
    'predictive utility, privacy, or clinical validity.'))

READMISSION 30-DAY LABEL DISTRIBUTION


,Count,Positive,Positive rate %
Real train,79567,9070,11.399
TabDDPM synthetic,79567,8407,10.566



MEAN PRIOR VISITS


,Real,TabDDPM
number_outpatient,0.365,0.411
number_emergency,0.200,0.209
number_inpatient,0.635,0.648



CATEGORY COVERAGE
race: real=6, TabDDPM=6
diag_3: real=20, TabDDPM=20
readmitted_30: real=2, TabDDPM=2

Run metadata saved: results/training/tabddpm_joint_100ep_run.json


**Initial interpretation:** TabDDPM generated **8,407 positive readmission cases** (10.566%), against **9,070** (11.399%) in real training data; the gap is **-0.833 percentage points**. The target was generated jointly, not sampled from a fixed class prior. This alone does **not** establish fidelity, predictive utility, privacy, or clinical validity.

## 8. Final experiment notes for the capstone

- **Training partition:** `train_semantic.csv` only; held-out test remains untouched.
- **Run type:** 100-epoch baseline, random seed 42, 200 diffusion steps, batch 1,024.
- **Feature handling:** 8 Gaussian + 37 multinomial features (including `readmitted_30`). Coded admission/discharge/source IDs remain categorical.
- **Key methodological deviation:** Unlike the original paper's class-conditional classification setup, this baseline uses **joint unconditional target generation** to assess whether the model itself learns readmission prevalence. Do not compare a class-balanced *conditional* output to an *unconditional* model as if both learned the same target distribution.
- **Checkpoints:** rolling checkpoint `models/tabddpm_joint_100ep_latest.pt` and final model `models/tabddpm_joint_100ep.pt`.
- **Output:** `data/synthetic/tabddpm_joint_100ep.csv` plus JSON run record.
- **Next:** evaluate TVAE, CTGAN, and TabDDPM with the same fidelity, TSTR utility, and privacy protocols. Do not publicly share row-level patient data; privacy risk must be checked before any synthetic-data release.